# Pandas基础

NumPy 处理的是同一种数堆在一起的情况。但真实数据是表格：一列是字符串，一列是整数，一列是日期，中间还夹着缺失值、重复行，经常要按某个字段分组求平均。这些用 ndarray 写起来很难受。

Pandas 只给两种结构：`Series`（一维）和 `DataFrame`（二维），剩下的全是围着这两样转。

## 1. Series 和 DataFrame

| | Series | DataFrame |
|---|---|---|
| 维度 | 一维 | 二维 |
| 组成 | 一组值 + 一个索引 | 若干列，每列是一个 Series，共用行索引 |
| 类比 | 带标签的数组 | 带行列标签的表格 / Excel 表 |
| 创建 | `pd.Series(...)` | `pd.DataFrame(...)` |

索引是 Pandas 和其他库最大的不同：每行、每列都有**标签**，可以按标签取，也可以按位置取。

In [1]:
import numpy as np
import pandas as pd

s = pd.Series([90, 85, 77], index=["Miku", "Rin", "Luka"], name="score")
print(s)

print(s.values)
print(s.index)
print(s["Miku"])
print(s.iloc[0])

Miku    90
Rin     85
Luka    77
Name: score, dtype: int64
[90 85 77]
Index(['Miku', 'Rin', 'Luka'], dtype='object')
90
90


不指定 index 时默认是 `RangeIndex`（0, 1, 2...），这也是为什么大部分时候 `df.loc[0]` 和 `df.iloc[0]` 一个结果，只是恰好对上了，换成别的索引就不一样了。索引还允许重复，重复索引取值会返回一组。

In [2]:
data = {
    "name": ["Miku", "Rin", "Luka", "Gumi", "Teto"],
    "class": ["A", "A", "B", "B", "C"],
    "score": [90, 85, 77, 92, 68],
    "age": [16, 15, 17, 16, 18],
}
df = pd.DataFrame(data)
print(df)
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

   name class  score  age
0  Miku     A     90   16
1   Rin     A     85   15
2  Luka     B     77   17
3  Gumi     B     92   16
4  Teto     C     68   18
(5, 4)
['name', 'class', 'score', 'age']
name     object
class    object
score     int64
age       int64
dtype: object


列名就是字典的 key，顺序在 Python 3.7 之后按插入顺序排。也可以用列表套字典的方式建，缺的字段自动补 `NaN`：

In [3]:
rows = [{"name": "Miku", "score": 90}, {"name": "Rin", "age": 15}]
print(pd.DataFrame(rows))

   name  score   age
0  Miku   90.0   NaN
1   Rin    NaN  15.0


但 Series 之间的运算**按索引对齐**，不是按位置。这点和 list、ndarray 完全不一样，第一次见会有点意外：

In [4]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20], index=["y", "z"])
print(a + b)      # 按索引对齐，x 对不上

x     NaN
y    12.0
z    23.0
dtype: float64


## 2. 读数据

真实项目里 90% 的时间是 `pd.read_xxx` 开头。`read_csv` 的参数最多，常用的几个：

| 参数 | 作用 |
|---|---|
| `filepath_or_buffer` | 路径、URL，或者文件对象 |
| `sep` | 分隔符，默认 `,`（中文 csv 有坑，见下） |
| `header` | 第几行当表头，`None` 表示没有表头 |
| `names` | 自己指定列名 |
| `index_col` | 把某一列设成行索引 |
| `usecols` | 只读某几列，大文件能省不少时间 |
| `dtype` | 指定列类型，比如 `{"id": str}` |
| `na_values` | 哪些字符串算缺失值 |
| `encoding` | 编码，中文文件常要 `"utf-8"` 或 `"gbk"` |

csv 里没有类型信息，Pandas 只能猜：全是数字的列就猜成整数，混了空值就变浮点，学号、邮编这种看着像数字的编号会被去掉前导 0。所以读进来第一件事是看类型。

In [5]:
from io import StringIO

csv_text = '''name,class,score,age
Miku,A,90,16
Rin,A,85,15
Luka,B,77,17
Gumi,B,92,
Teto,C,68,18
'''

df = pd.read_csv(StringIO(csv_text))
print(df)
print(df.dtypes)

   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
2  Luka     B     77  17.0
3  Gumi     B     92   NaN
4  Teto     C     68  18.0
name      object
class     object
score      int64
age      float64
dtype: object


`age` 那列因为有空格子，被读成了 `float64`，本来应该是整数。`dtype` 可以强制指定，或者读完用 `astype` 转。

写回文件是 `to_csv`，`index=False` 表示不要把行索引也写进去（不然会多一列没名字的 0,1,2...）：

In [6]:
print(df.to_csv(index=False))

name,class,score,age
Miku,A,90,16.0
Rin,A,85,15.0
Luka,B,77,17.0
Gumi,B,92,
Teto,C,68,18.0



其他格式一行就能换：`read_excel`、`read_json`、`read_sql`、`read_html`。Excel 需要装 `openpyxl`，读中文 csv 遇到乱码先试试 `encoding="utf-8-sig"`。

## 3. 先看清楚数据

拿到一份表别急着写逻辑，先看它的形状和类型。

| 方法 | 看什么 |
|---|---|
| `df.head(n)` / `tail(n)` | 前 / 后几行 |
| `df.sample(n)` | 随机几行，比 head 更能反映真实情况 |
| `df.shape` | 几行几列 |
| `df.dtypes` | 每列类型 |
| `df.info()` | 行数、每列非空数量、类型、占多少内存 |
| `df.describe()` | 数值列的 count / mean / std / min / 分位数 / max |
| `df.isnull().sum()` | 每列缺几个 |
| `df["col"].value_counts()` | 某列各取值出现的次数 |

In [7]:
print(df.head(3))
print(df.tail(2))
print(df.sample(2, random_state=0))

   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
2  Luka     B     77  17.0
   name class  score   age
3  Gumi     B     92   NaN
4  Teto     C     68  18.0
   name class  score   age
2  Luka     B     77  17.0
0  Miku     A     90  16.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    5 non-null      object 
 1   class   5 non-null      object 
 2   score   5 non-null      int64  
 3   age     4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 288.0+ bytes


In [9]:
print(df.describe())
print(df.describe().T)

           score        age
count   5.000000   4.000000
mean   82.400000  16.500000
std     9.914636   1.290994
min    68.000000  15.000000
25%    77.000000  15.750000
50%    85.000000  16.500000
75%    90.000000  17.250000
max    92.000000  18.000000
       count  mean       std   min    25%   50%    75%   max
score    5.0  82.4  9.914636  68.0  77.00  85.0  90.00  92.0
age      4.0  16.5  1.290994  15.0  15.75  16.5  17.25  18.0


`describe` 默认只统计数值列。想看类别列就加 `include="object"`，不过我更常用 `value_counts`。

注意 `describe` 的 `count`：如果某列 count 比行数少，说明有缺失，顺便就看出来了。

In [10]:
print(df["class"].value_counts())
print(df["score"].mean(), df["score"].median())
print(df.isnull().sum())
print(df.nunique())
print(df.corr(numeric_only=True).round(3))

class
A    2
B    2
C    1
Name: count, dtype: int64
82.4 85.0
name     0
class    0
score    0
age      1
dtype: int64
name     5
class    3
score    5
age      4
dtype: int64
       score    age
score  1.000 -0.858
age   -0.858  1.000


## 4. 选数据

取数据主要靠下面几种写法，别混：

| 写法 | 选什么 | 例子 |
|---|---|---|
| `df["col"]` | 一列，返回 `Series` | `df["score"]` |
| `df[["a", "b"]]` | 多列，返回 `DataFrame`（注意是双括号） | `df[["name", "score"]]` |
| `df.loc[行标签, 列标签]` | 按**标签**取 | `df.loc[0, "score"]` |
| `df.iloc[行位置, 列位置]` | 按**位置**取 | `df.iloc[0, 2]` |

`loc` 的切片**两端都包含**，`iloc` 和 Python 一样左闭右开。

In [11]:
print(df["score"])
print(df[["name", "score"]])

print(df.loc[0, "score"])
print(df.iloc[0, 2])

print(df.loc[0:2, "name":"score"])   # loc 切片两端都含
print(df.iloc[0:2, 0:2])

0    90
1    85
2    77
3    92
4    68
Name: score, dtype: int64
   name  score
0  Miku     90
1   Rin     85
2  Luka     77
3  Gumi     92
4  Teto     68
90
90
   name class  score
0  Miku     A     90
1   Rin     A     85
2  Luka     B     77
   name class
0  Miku     A
1   Rin     A


只取一个格子也可以用 `df.at[0, "score"]` 和 `df.iat[0, 2]`，比 loc / iloc 快，但初学不用管，知道有就行。

#### 布尔筛选

条件筛选和 NumPy 一样，`&`、`|` 要加括号。筛出来的结果是个新表，**改它不会改原表**，想在原表上改要用 `.loc`：

```python
df.loc[df["score"] < 60, "class"] = "待补考"     # 对
df[df["score"] < 60]["class"] = "待补考"          # 错，只改了临时副本
```

第二种还会报 `SettingWithCopyWarning`，看到这个警告基本就是上面这种情况。

In [12]:
print(df[df["score"] > 80])
print(df[(df["class"] == "A") | (df["age"] > 17)])
print(df[df["class"].isin(["A", "C"])])
print(df[df["score"].notnull()])

   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
3  Gumi     B     92   NaN
   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
4  Teto     C     68  18.0
   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
4  Teto     C     68  18.0
   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
2  Luka     B     77  17.0
3  Gumi     B     92   NaN
4  Teto     C     68  18.0


#### 设置索引 / 还原索引

把某一列设成索引之后，`loc` 就能按它取，语义清楚很多。反过来用 `reset_index()` 还原。

In [13]:
df2 = df.set_index("name")
print(df2.loc["Miku"])
print(df2.reset_index().head(2))

class       A
score      90
age      16.0
Name: Miku, dtype: object
   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0


## 5. 缺失值

表格数据里最烦的是缺失值和重复行。

缺失值在 Pandas 里显示成 `NaN`（时间类型是 `NaT`），本质是浮点的 `nan`。所以一列本来是整数的，只要有一个空就整列变成 `float64`，刚才的 `age` 就是这样。

判断只能用 `isnull()` / `notnull()`，**不能写 `== np.nan`**：nan 不等于任何东西，包括它自己。

In [14]:
print(df["age"].isnull())
print(df["age"].isnull().sum())
print(np.nan == np.nan)

0    False
1    False
2    False
3     True
4    False
Name: age, dtype: bool
1
False


处理就两条路：删或者填。

In [15]:
print(df.dropna().shape)
print(df.dropna(subset=["score"]).shape)

print(df["age"].fillna(df["age"].mean()))
print(df["age"].ffill())

(4, 4)
(5, 4)
0    16.0
1    15.0
2    17.0
3    16.5
4    18.0
Name: age, dtype: float64
0    16.0
1    15.0
2    17.0
3    17.0
4    18.0
Name: age, dtype: float64


填什么其实是个业务问题：成绩缺了填 0 是错的（0 分会把平均分带偏），填平均分勉强能用，填未知又得把列改成字符串。删也一样，删掉的可能是最有价值的那几行。

自己练手的时候先跑通，真正做项目的时候这一步要单独写个说明。

#### 重复值

In [16]:
d2 = pd.concat([df, df.iloc[[0]]])
print(d2.shape)
print(d2.duplicated().sum())

print(d2.drop_duplicates().shape)
print(d2.drop_duplicates(subset=["class"], keep="first"))

(6, 4)
1
(5, 4)
   name class  score   age
0  Miku     A     90  16.0
2  Luka     B     77  17.0
4  Teto     C     68  18.0


## 6. 改列、加列、删列

新增列就是赋值，长度要么和行数一致，要么给一个标量（会自动广播到每一行）。想按行算就用 `apply(axis=1)`，想按值替换就用 `map`。

In [17]:
d = df.copy()

d["pass"] = d["score"] >= 80
d["score2"] = d["score"] * 1.1
d["level"] = d["score"].apply(lambda x: "high" if x >= 85 else "low")
d["class_cn"] = d["class"].map({"A": "一班", "B": "二班", "C": "三班"})

print(d)

   name class  score   age   pass  score2 level class_cn
0  Miku     A     90  16.0   True    99.0  high       一班
1   Rin     A     85  15.0   True    93.5  high       一班
2  Luka     B     77  17.0  False    84.7   low       二班
3  Gumi     B     92   NaN   True   101.2  high       二班
4  Teto     C     68  18.0  False    74.8   low       三班


`apply` 本质是一行行（或一列列）调 Python 函数，速度一般，数据大的时候能向量化就向量化。简单条件用 `np.where`，连续值分段用 `pd.cut`，都比 apply 快也清楚：

In [18]:
d["level2"] = np.where(d["score"] >= 85, "高", "低")
d["band"] = pd.cut(d["score"], bins=[0, 60, 80, 100], labels=["不及格", "及格", "优秀"])

print(d[["score", "level", "level2", "band"]])

   score level level2 band
0     90  high      高   优秀
1     85  high      高   优秀
2     77   low      低   及格
3     92  high      高   优秀
4     68   low      低   及格


删列用 `drop(columns=[...])`，改名用 `rename(columns={...})`。这些操作默认都**返回新表，不动原表**；想就地改要么加 `inplace=True`，要么把返回值赋回去。我一般不用 `inplace`，写 `d = d.drop(...)` 更直观，出了问题也好回退。

In [19]:
d3 = d.drop(columns=["score2", "level2"])
print(d3.columns.tolist())
print(d3.rename(columns={"score": "成绩"}).columns.tolist())

['name', 'class', 'score', 'age', 'pass', 'level', 'class_cn', 'band']
['name', 'class', '成绩', 'age', 'pass', 'level', 'class_cn', 'band']


## 7. 排序和排名

In [20]:
print(df.sort_values("score", ascending=False))
print(df.sort_values(["class", "score"], ascending=[True, False]))
print(df.sort_index(ascending=False))
print(df["score"].rank(ascending=False))

   name class  score   age
3  Gumi     B     92   NaN
0  Miku     A     90  16.0
1   Rin     A     85  15.0
2  Luka     B     77  17.0
4  Teto     C     68  18.0
   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
3  Gumi     B     92   NaN
2  Luka     B     77  17.0
4  Teto     C     68  18.0
   name class  score   age
4  Teto     C     68  18.0
3  Gumi     B     92   NaN
2  Luka     B     77  17.0
1   Rin     A     85  15.0
0  Miku     A     90  16.0
0    2.0
1    3.0
2    4.0
3    1.0
4    5.0
Name: score, dtype: float64


`rank` 给的是名次（1 是最小，`ascending=False` 时 1 是最大），并列时默认取平均名次，想按出现顺序排可以给 `method="first"`。

## 8. 分组聚合

`groupby` 和 SQL 的 `GROUP BY` 一个意思：先按某列分堆，再对每一堆做同样的计算。写的时候分三步：`groupby` 分、选列、聚合。

In [21]:
print(df.groupby("class")["score"].mean())
print(df.groupby("class").agg({"score": ["mean", "max"], "age": "mean"}))
print(df.groupby("class").size())

class
A    87.5
B    84.5
C    68.0
Name: score, dtype: float64
      score       age
       mean max  mean
class                
A      87.5  90  15.5
B      84.5  92  17.0
C      68.0  68  18.0
class
A    2
B    2
C    1
dtype: int64


`agg` 可以一次给多个列、多个函数，结果会自动变成多级列名，读的时候用 `df.columns` 看一眼。

`transform` 和 `agg` 不一样：它返回和原表**一样长**的结果，所以能直接接成一列。组内标准化（每个班内部各自减均值除标准差）就用它，处理不同班级考试难度不同这类问题很方便。

In [22]:
d["score_z"] = df.groupby("class")["score"].transform(lambda s: (s - s.mean()) / s.std())
print(d[["name", "class", "score", "score_z"]])

   name class  score   score_z
0  Miku     A     90  0.707107
1   Rin     A     85 -0.707107
2  Luka     B     77 -0.707107
3  Gumi     B     92  0.707107
4  Teto     C     68       NaN


In [23]:
print(df.pivot_table(index="class", values="score", aggfunc="mean"))

       score
class       
A       87.5
B       84.5
C       68.0


`pivot_table` 就是分组再摆成表格，两个分类变量交叉的时候比 groupby 好用，后面画热力图也常用它。

## 9. 拼表

两个东西容易混：

| | 方向 | 用途 |
|---|---|---|
| `pd.concat` | 上下叠（axis=0）或左右拼（axis=1） | 同一份结构的数据分批读进来，拼成一整份 |
| `df.merge` | 按某列对齐，像 SQL 的 join | 两张表有关联字段，想把信息并到一起 |

`merge` 的 `how` 和 SQL 一样：`inner`（默认，取交集）、`left`、`right`、`outer`。

In [24]:
info = pd.DataFrame({
    "class": ["A", "B", "C"],
    "teacher": ["Kagamine", "Megurine", "Hatsune"],
})

print(df.merge(info, on="class", how="left"))

   name class  score   age   teacher
0  Miku     A     90  16.0  Kagamine
1   Rin     A     85  15.0  Kagamine
2  Luka     B     77  17.0  Megurine
3  Gumi     B     92   NaN  Megurine
4  Teto     C     68  18.0   Hatsune


In [25]:
s1 = df[["name", "score"]]
s2 = df[["age"]]
print(pd.concat([s1, s2], axis=1).head(3))
print(pd.concat([df.head(2), df.tail(2)], ignore_index=True))

   name  score   age
0  Miku     90  16.0
1   Rin     85  15.0
2  Luka     77  17.0
   name class  score   age
0  Miku     A     90  16.0
1   Rin     A     85  15.0
2  Gumi     B     92   NaN
3  Teto     C     68  18.0


> `merge` 一条都合不上，先怀疑两边 `on` 列的类型不一样：一边是 `"1"` 另一边是 `1`，看着一样其实永远不相等。`df.dtypes` 一眼就能看出来。

## 10. 时间和字符串

这两块内容很多，这里只记最常用的：转成时间类型之后可以按年 / 月 / 星期取，两个时间相减得到 `Timedelta`。

In [26]:
t = pd.to_datetime(["2026-03-01", "2026-03-05", "2026-04-10"])
print(t)
print(t.year.tolist(), t.month.tolist(), t.dayofweek.tolist())   # 0 是周一
print((t[2] - t[0]).days)

DatetimeIndex(['2026-03-01', '2026-03-05', '2026-04-10'], dtype='datetime64[ns]', freq=None)
[2026, 2026, 2026] [3, 3, 4] [6, 3, 4]
40


字符串列要加 `.str` 才能用 Python 的字符串方法，好处是遇到 `NaN` 会自动跳过，不会报错。

In [27]:
s = pd.Series(["Miku", " miku ", "RIN"])
print(s.str.strip().str.lower().tolist())
print(s.str.contains("mi", case=False).tolist())
print(s.str.len().tolist())

['miku', 'miku', 'rin']
[True, True, False]
[4, 6, 3]


## 11. 和 sklearn 接起来

Pandas 负责洗数据，NumPy / sklearn 负责算，中间那一步是 `to_numpy()`。一个表格任务的顺序大概是：

1. `read_csv` 读进来
2. `info()` + `describe()` 看清数据
3. 处理缺失、重复、异常值
4. 类别列转数值（独热 / 有序编码）
5. 拆成 `X`（特征）和 `y`（标签）
6. `to_numpy()` 交给模型

In [28]:
d = df.copy()
d["age"] = d["age"].fillna(d["age"].mean())

X_cat = pd.get_dummies(d[["class"]], prefix="class").astype(int)
X = pd.concat([d[["age"]], X_cat], axis=1)
y = d["score"]

print(X)
print(X.shape, y.shape)

    age  class_A  class_B  class_C
0  16.0        1        0        0
1  15.0        1        0        0
2  17.0        0        1        0
3  16.5        0        1        0
4  18.0        0        0        1
(5, 4) (5,)


`get_dummies` 现在默认给 `bool`，所以顺手 `astype(int)` 一下，不然有些模型会嫌类型奇怪。

其实现在的 sklearn 直接吃 DataFrame 也可以，列名对得上就行。但初学阶段我建议自己手动转一次 numpy，能看清每一列到底是什么、顺序对不对。